# 26 | LangChain Middleware：state 和 runtime 到底是什么

上一篇讲 `request / handler`：它们负责“包住一次模型或工具调用”。

这一篇讲 `state / runtime`：它们不负责放行调用，而是让 Middleware 看清楚 Agent 当前跑到哪了，以及这次运行带了哪些外部信息。

一句话：

`state` 看 Agent 内部状态，`runtime` 看本次运行环境。

## 一、先分清楚它们

`before_model` 会在每次模型调用前执行，`after_model` 会在每次模型返回后执行。

所以这两个 hook 里常见的是：

- `state`：Agent 当前累积出来的状态，最常看的是 `state["messages"]`；
- `runtime`：本次运行环境，最常用的是 `runtime.context`。

比如用户说：“我要导出 20 万行订单报表”。

用户消息里不一定写着请求 ID、用户角色、所属区域，但系统侧通常知道这些信息。它们可以放进 `runtime.context`，供 Middleware 记录和判断。注意：`runtime.context` 不会自动变成模型可见的用户消息，也不会自动填进工具参数。

In [ ]:
import os
from typing import Any, Literal, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing_extensions import NotRequired

from langchain.agents import create_agent
from langchain.agents.middleware import AgentState, after_model, before_model
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.runtime import Runtime

load_dotenv(dotenv_path=".env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)

## 二、定义 state 和 context

这里用一个很小的例子：记录模型调用次数，同时从运行上下文里读取请求 ID 和用户角色。

In [ ]:
class AuditState(AgentState):
    # AgentState 是 LangChain Agent 默认状态，里面已经有 messages。
    # 继承它，是为了在默认状态上加一个 model_call_count。
    # NotRequired 表示调用 agent.invoke(...) 时可以不传，后面 Middleware 会自己维护。
    model_call_count: NotRequired[int]


class RunContext(TypedDict):
    # TypedDict 用来声明 runtime.context 的结构。
    # 这些信息来自业务系统，不属于用户聊天内容。
    request_id: str  # 请求 ID，方便日志追踪。
    user_role: str   # 用户角色，比如 finance / operation。

In [ ]:
class ExportRiskInput(BaseModel):
    report_name: str = Field(description="报表名称，例如 order_export")
    estimated_rows: int = Field(description="预计导出的数据行数")


@tool(args_schema=ExportRiskInput)
def estimate_export_risk(report_name: str, estimated_rows: int) -> dict:
    """评估订单报表导出风险。"""
    risk_level = "high" if estimated_rows > 100_000 else "normal"
    return {
        "risk_level": risk_level,
        "suggestion": "需要审批或拆分导出" if risk_level == "high" else "可以正常导出",
    }


## 三、在模型调用前后看状态

下面两个 Middleware 只做一件事：打印关键字段，不打印一大坨完整对象。

In [ ]:
@before_model(state_schema=AuditState)
def log_before_model(state: AuditState, runtime: Runtime[RunContext]) -> None:
    context = runtime.context or {}

    print("[before_model]")
    print(f"messages={len(state['messages'])}")
    print(f"model_call_count={state.get('model_call_count', 0)}")
    print(f"request_id={context.get('request_id')}")
    print(f"user_role={context.get('user_role')}")
    print("=" * 40)


@after_model(state_schema=AuditState)
def log_after_model(state: AuditState, runtime: Runtime[RunContext]) -> dict[str, Any]:
    latest_message = state["messages"][-1]
    tool_calls = getattr(latest_message, "tool_calls", [])

    print("[after_model]")
    print(f"has_tool_calls={bool(tool_calls)}")
    if tool_calls:
        print(f"tool_names={[item['name'] for item in tool_calls]}")
    print("=" * 40)

    # 返回 dict 会更新 Agent state。
    return {"model_call_count": state.get("model_call_count", 0) + 1}

`before_model` 只是观察，所以不用返回内容。

`after_model` 返回了一个 `dict`，LangChain 会把它合并回 Agent state。这里就是把 `model_call_count` 加 1。

In [ ]:
agent = create_agent(
    model=model,
    tools=[estimate_export_risk],
    middleware=[log_before_model, log_after_model],
    context_schema=RunContext,
    system_prompt=(
        "你是订单报表导出助手。"
        "当用户询问大数据量导出风险时，必须调用 estimate_export_risk。"
        "用户角色由系统上下文记录，不需要向用户追问。"
    ),
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "我要导出 order_export，预计 200000 行，帮我判断是否有风险。",
            }
        ],
        # model_call_count 是自定义 state 字段，来自 AuditState。
        # 它不是用户消息，也不是模型参数，不会作为内容发给模型。
        # 这里给初始值 0，后面的 after_model 会返回 dict，把它累加更新到 Agent state。
        # 如果没有声明 state_schema=AuditState，随便塞自定义字段就不一定能被正确保留。
        "model_call_count": 0,
    },
    context={
        "request_id": "REQ-2026-0608-001",
        "user_role": "finance",
    },
)

print("\n[最终回复]")
print(result["messages"][-1].content)
print("\n[模型调用次数]")
print(result.get("model_call_count"))

## 四、怎么看输出

如果模型调用了工具，通常会看到两轮模型调用：

1. 第一次 `before_model`：准备让模型理解用户问题；
2. 第一次 `after_model`：模型决定调用 `estimate_export_risk`；
3. 工具执行，返回风险评估结果；
4. 第二次 `before_model`：带着工具结果再问模型；
5. 第二次 `after_model`：模型生成最终回复。

所以最后 `model_call_count` 通常是 2。

这里的重点不是“统计次数”本身，而是说明：`state` 会随着 Agent 执行不断变化，Middleware 可以在节点前后看到这些变化。

另外，`runtime.context` 里的 `user_role` 只是给 Middleware 读取。除非你主动把它写进提示词、工具参数或 state，否则模型不会自动知道它。

## 五、别和 request / handler 混了

| 概念 | 用在哪里 | 重点 |
|---|---|---|
| `request` | `wrap_model_call` / `wrap_tool_call` | 这一次模型或工具调用准备带什么 |
| `handler` | `wrap_model_call` / `wrap_tool_call` | 要不要继续交给下游执行 |
| `state` | `before_model` / `after_model` | Agent 当前累积出来的状态 |
| `runtime` | `before_model` / `after_model` | 本次运行传进来的上下文和能力 |

`request / handler` 更像拦截器；`state / runtime` 更像仪表盘。

仪表盘不用装太多，关键指标清楚就行。否则 Agent 没累，读日志的人先累了。

## 小结

`state` 是 Agent 内部状态，最常看 `messages`，也可以通过继承 `AgentState` 加自定义字段。

`runtime` 是本次运行环境，常用 `runtime.context` 读取业务信息，比如请求 ID、用户角色。

`before_model` 适合模型调用前观察上下文；`after_model` 适合模型返回后检查结果，并按需更新 state。